# Module install and load

In [ ]:
!pip install scikit-image

In [ ]:
!pip install pillow numpy pandas

# Module import

In [ ]:
# 1. Import Required Libraries
import os
from PIL import Image, ImageStat
import numpy as np
import pandas as pd
from skimage.transform import radon
from skimage.feature import canny
import scipy.stats
import matplotlib.pyplot as plt
import re
from pathlib import Path


# List image files

In [ ]:


image_folder = Path(r"C:\Users\Patrick\Gutbrain_R\data\food")
image_files = [f.name for f in image_folder.iterdir() if f.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}]
print(f"Found {len(image_files)} images.")

# Functions

## Function to calculate visual properties per image

In [ ]:


## 3. Calculate visual properties for each image
def calculate_image_properties(image_path):
    with Image.open(image_path) as img:
        img = img.convert('RGB')
        np_img = np.array(img)
        size = img.size

        # Mask: True for object pixels (not black)
        custom_masks = {
            "HiSaltPretzels.bmp": [0, 4],
            "HiSweetm&mamandes.bmp": [0, 1],
            "HiSweetReese.bmp": [0, 2]
        }

        filename = os.path.basename(image_path)
        gray_img = np.dot(np_img[...,:3], [0.2989, 0.5870, 0.1140]).astype(int)

        if filename in custom_masks:
            # Mask: True for object pixels (not background gray levels)
            mask = ~np.isin(gray_img, custom_masks[filename])
        else:
            # Default: True for object pixels (not almost black)
            mask = np.any(np_img > 1, axis=-1)

        object_pixels = np_img[mask]
        num_object_pixels = object_pixels.shape[0]
        total_pixels = size[0] * size[1]

        if num_object_pixels > 0:
            mean_color = np.round(object_pixels.mean(axis=0), 2)

            gray_object_pixels = np.dot(object_pixels[...,:3], [0.2989, 0.5870, 0.1140])
            brightness = np.round(gray_object_pixels.mean(), 2)
            contrast = np.round(gray_object_pixels.std(), 2)

            
        else:
            mean_color = [np.nan, np.nan, np.nan]
            brightness = np.nan
            contrast = np.nan


        return {
            'filename': os.path.basename(image_path),
            'width': size[0],
            'height': size[1],
            'mean_color_r': mean_color[0],
            'mean_color_g': mean_color[1],
            'mean_color_b': mean_color[2],
            'brightness': brightness,
            'contrast': contrast,
            'num_object_pixels': num_object_pixels

        }
    


# Compute visual properties and analyze them

## Run function on images and describe output table

In [ ]:
results = [calculate_image_properties(os.path.join(image_folder, f)) for f in image_files]
df = pd.DataFrame(results)

numeric_cols = df.select_dtypes(include=[np.number]).columns
zscore_df = df[numeric_cols].apply(scipy.stats.zscore, nan_policy='omit')
zscore_df = zscore_df.add_suffix('_zscore')

# Concatenate z-scores to the original DataFrame
df = pd.concat([df, zscore_df], axis=1)

## 4. Display summary statistics
df.describe()

## Show the table of visual properties

In [ ]:
df.head(50)

# Merge with updated pricing

## upload pricing csv to df

In [ ]:
pricing_path = Path(r"C:\Users\Patrick\Gutbrain_R\data\kcal_real_v3.csv")
pricing_df = pd.read_csv(pricing_path, sep=",")

## Clean the pricing df

In [ ]:
# Drop columns only if they exist in pricing_df
cols_to_keep = ['picture', 'Moyenne Densite'] + list(pricing_df.columns[-5:])
pricing_df = pricing_df[cols_to_keep]
pricing_df.head()

pricing_df['picture'] = pricing_df['picture'].str.replace(r'^Food/', '', regex=True)


## merge pricing and visual properties df

In [ ]:
pricing_df['picture'] = (
    pricing_df['picture']
    .astype(str)
    .str.lower()
    .str.replace(r'^food/', '', regex=True)
    .str.replace(r'\..*$', '', regex=True)
)

df['filename'] = (
    df['filename']
    .astype(str)
    .str.lower()
    .str.replace(r'\..*$', '', regex=True)
)

merged_df = pd.merge(
    pricing_df,
    df,
    left_on='picture',
    right_on='filename',
    how='inner'
)

In [ ]:
merged_df

## Open the df_grouped_by_picture

In [ ]:
df_grouped_by_picture = pd.read_csv('C:\\Users\\Patrick\\Gutbrain_R\\data\\df_grouped_by_picture.csv', sep=',')


## set up for the merge

In [ ]:
# Standardize 'picture' columns for merging

def clean_picture(s):
    s = s.lower()
    s = re.sub(r'^(lo|hisweet|hisalt)', '', s)
    s = re.sub(r'\..*$', '', s)
    s = s.strip('_')
    if s == 'juliennes':
        s = 'julienne'
    return s

merged_df['picture_clean'] = merged_df['picture'].apply(clean_picture)
df_grouped_by_picture['picture_clean'] = df_grouped_by_picture['picture'].apply(clean_picture)

# Use left join to keep all rows from df_grouped_by_picture
final_merged_df = pd.merge(df_grouped_by_picture, merged_df, on='picture_clean', how='left')


# Use the correct column name for missing
missing = final_merged_df[final_merged_df.isnull().any(axis=1)]['picture_x']


In [ ]:
print("Cleaned names in df_grouped_by_picture:")
print(sorted(df_grouped_by_picture['picture_clean'].unique()))
print("\nCleaned names in merged_df:")
print(sorted(merged_df['picture_clean'].unique()))

In [ ]:
print(final_merged_df.describe())
print("\nNaN count per column:")
print(final_merged_df.isna().sum())

# save df of stimuli properties

In [ ]:
final_merged_df.to_csv('C:\\Users\\Patrick\\Gutbrain_R\\data\\gutbrain_food_final_merged_df.tsv', sep='\t', index=False)
print("Saved to C:\\Users\\Patrick\\Gutbrain_R\\data\\gutbrain_food_final_merged_df.tsv")